In [1]:
import requests
import pandas as pd
from datetime import datetime, date, timezone, timedelta
import json
import time
import random
from typing import Any
import json
from pathlib import Path
import re

from renewables_permitting.utils import as_list, save_parquet, normalize_text

BASE_URL = "https://www.boe.es/datosabiertos/api/boe/sumario"

# PROJECT_ROOT = Path(__file__).resolve().parents[2]  # fuera del notebook
PROJECT_ROOT = Path.cwd().parent  # dentro del notebook

DATA_DIR = PROJECT_ROOT / "data"

BRONZE_DIR = DATA_DIR / "bronze"
SILVER_DIR = DATA_DIR / "silver"
GOLD_DIR = DATA_DIR / "gold"

In [ ]:
boe_items = pd.read_parquet(
    SILVER_DIR / "boe_items" / "boe_items_normalized.parquet"
)

,identificador,control,titulo,url_html,url_xml,url_pdf,pdf_size_bytes,pdf_size_kbytes,pagina_inicial,pagina_final,...,departamento_nombre,epigrafe_nombre,item_location,source,country,bronze_date,seccion_nombre_norm,departamento_nombre_norm,epigrafe_nombre_norm,titulo_norm
1881,BOE-A-2024-9608,2024/7789,"Resolución de 6 de mayo de 2024, de la Direcci...",https://www.boe.es/diario_boe/txt.php?id=BOE-A...,https://www.boe.es/diario_boe/xml.php?id=BOE-A...,https://www.boe.es/boe/dias/2024/05/13/pdfs/BO...,239401,234,54419,54425,...,MINISTERIO PARA LA TRANSICIÓN ECOLÓGICA Y EL R...,Impacto ambiental,departamento.epigrafe.item,boe,ES,20240513,iii otras disposiciones,ministerio para la transicion ecologica y el r...,impacto ambiental,resolucion de 6 de mayo de 2024 de la direccio...


# Filtrado determinista

In [32]:
# TODO: ajustar keywords y filtrado

keywords_departamento = [
    "energia",
    "industria",
    "transicion ecologica",
    "movilidad",
    "medio ambiente",
    "politica territorial",
    "transportes"
]

keywords_epigrafe = [
    "energia electrica",
    "impacto ambiental",
    "instalaciones electricas",
    "otros anuncios oficiales"
]

keywords_titulo = [
    "almacenamiento",
    "autorizacion administrativa de construccion",
    "autorizacion administrativa previa",
    "bateria",
    "declaracion de impacto ambiental",
    "energia electrica",
    "energia solar",
    "instalacion solar",
    "eolic",
    "evacuacion",
    "fotovoltaic",
    "hibridacion",
    "impacto ambiental",
    "afeccion ambiental",
    "informacion publica",
    "infraestructura de evacuacion",
    "linea de evacuacion",
    "linea electrica",
    "parque eolico",
    "planta fotovoltaica",
    "planta solar",
    "plantas solares",
    "repotenciacion",
    "solar termica",
    "subestacion",
    "utilidad publica",
]

# negative_keywords = [
#     "carreteras",
#     "ferrocarriles",
#     "aeropuerto",
#     "puerto",
#     "concesión de aguas",
#     "patrimonio cultural",
# ]

In [33]:
pattern_departamento = "|".join(
    re.escape(k) for k in keywords_departamento
)

pattern_epigrafe = "|".join(
    re.escape(k) for k in keywords_epigrafe
)

pattern_titulo = "|".join(
    re.escape(k) for k in keywords_titulo
)

In [34]:
mask_departamento = (
    boe_items["departamento_nombre_norm"]
    .str.contains(pattern_departamento, regex=True, na=False)
)

mask_epigrafe = (
    boe_items["epigrafe_nombre_norm"]
    .str.contains(pattern_epigrafe, regex=True, na=False)
)

mask_titulo = (
    boe_items["titulo_norm"]
    .str.contains(pattern_titulo, regex=True, na=False)
)

# mask =  mask_departamento | mask_epigrafe | mask_titulo
mask = mask_titulo

boe_candidates_normalized = boe_items.loc[mask].copy()

# boe_candidates_normalized.loc[boe_candidates_normalized["titulo_norm"].notnull(), "titulo_norm"].values[0:10]

In [35]:
# Añadir idenificador único para cada documento, que se usará para nombrar los archivos de texto y relacionarlos con el resto de la información. 
# El formato será "YYYYMMDD_identificador", donde "identificador" es el campo "identificador" del BOE, que es único para cada documento.

boe_candidates_normalized["doc_file_stem"] = (
    pd.to_datetime(boe_candidates_normalized["fecha_publicacion"]).dt.strftime("%Y%m%d")
    + "_"
    + boe_candidates_normalized["identificador"]
)

cols = [
    "identificador",
    "doc_file_stem",
    "titulo",
    "fecha_publicacion",
    "year",
    "month",
    "day",
    "url_html",
    "url_xml",
    "url_pdf",
    "pdf_size_bytes",
    "pdf_size_kbytes",
    "pagina_inicial",
    "pagina_final",
    "diario_numero",
    "seccion_codigo",
    "seccion_nombre",
    "departamento_codigo",
    "departamento_nombre",
    "epigrafe_nombre",
    "control",
    "item_location",
    "source",
    "country",
    "bronze_date",
    "seccion_nombre_norm",
    "departamento_nombre_norm",
    "epigrafe_nombre_norm",
    "titulo_norm",
]

boe_candidates_normalized = boe_candidates_normalized[cols]

In [36]:
save_parquet(
    boe_candidates_normalized,
    SILVER_DIR / "boe_candidates" / "boe_candidates_normalized.parquet",
)

# Pruebas

In [37]:
test = pd.read_parquet(
    SILVER_DIR / "boe_candidates" / "boe_candidates_normalized.parquet"
)

test.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1266 entries, 0 to 1265
Data columns (total 29 columns):
 #   Column                    Non-Null Count  Dtype 
---  ------                    --------------  ----- 
 0   identificador             1266 non-null   object
 1   doc_file_stem             1266 non-null   object
 2   titulo                    1266 non-null   object
 3   fecha_publicacion         1266 non-null   object
 4   year                      1266 non-null   int64 
 5   month                     1266 non-null   int64 
 6   day                       1266 non-null   int64 
 7   url_html                  1266 non-null   object
 8   url_xml                   1266 non-null   object
 9   url_pdf                   1266 non-null   object
 10  pdf_size_bytes            1266 non-null   object
 11  pdf_size_kbytes           1266 non-null   object
 12  pagina_inicial            1266 non-null   object
 13  pagina_final              1266 non-null   object
 14  diario_numero           

In [10]:
# test.loc[test["epigrafe_nombre"].isnull()]

In [30]:
test[1:2].values

array([['BOE-B-2021-32555', '20210707_BOE-B-2021-32555',
        'Resolución de la Dirección General de Planificación y Evaluación de la Red Ferroviaria por la que se abre Información Pública correspondiente al Expediente de Expropiación Forzosa 305ADIF2104 motivado por las obras del "Proyecto de Construcción para la Implantación del Ancho Estándar en el Corredor Mediterráneo. Tramo: Castellbisbal-Murcia. Subtramo: Vinaroz-Vandellós. Vía y Electrificación", en los términos municipales de Camarles, Freginals, L’Aldea, L’Ametlla de Mar y Ulldecona (Tarragona) y Vinarós (Castellón).',
        '2021-07-07', 2021, 7, 7,
        'https://www.boe.es/diario_boe/txt.php?id=BOE-B-2021-32555',
        'https://www.boe.es/diario_boe/xml.php?id=BOE-B-2021-32555',
        'https://www.boe.es/boe/dias/2021/07/07/pdfs/BOE-B-2021-32555.pdf',
        '261121', '255', '43054', '43056', '161', '5B',
        'V. Anuncios. - B. Otros anuncios oficiales', '9572',
        'MINISTERIO DE TRANSPORTES, MOVILIDAD

In [38]:
test.loc[test["identificador"].isin(["BOE-A-2024-9608"])]

,identificador,doc_file_stem,titulo,fecha_publicacion,year,month,day,url_html,url_xml,url_pdf,...,epigrafe_nombre,control,item_location,source,country,bronze_date,seccion_nombre_norm,departamento_nombre_norm,epigrafe_nombre_norm,titulo_norm
142,BOE-A-2024-9608,20240513_BOE-A-2024-9608,"Resolución de 6 de mayo de 2024, de la Direcci...",2024-05-13,2024,5,13,https://www.boe.es/diario_boe/txt.php?id=BOE-A...,https://www.boe.es/diario_boe/xml.php?id=BOE-A...,https://www.boe.es/boe/dias/2024/05/13/pdfs/BO...,...,Impacto ambiental,2024/7789,departamento.epigrafe.item,boe,ES,20240513,iii otras disposiciones,ministerio para la transicion ecologica y el r...,impacto ambiental,resolucion de 6 de mayo de 2024 de la direccio...
